In [9]:
import numpy as np
import sys
sys.path.append("../../")
sys.path.append("../../Visualization/")
sys.path.append("../../../")

In [10]:
sys.path.append("../../experiments/parametrization_experiments/")

In [11]:
import parametrization_experiment_helper

In [12]:
import pandas as pd
import json
import matplotlib.pyplot as plt
import numpy as np

In [39]:
import visualize_stiffness
import importlib
importlib.reload(visualize_stiffness)
from scipy.spatial import ConvexHull, convex_hull_plot_2d


In [40]:
kappa_path = None
import json

In [50]:
data_collection = []
labels = []
for pattern in parametrization_experiment_helper.Pattern_data:
    experiment_file = pattern['experiment_file']
    stiffness_path = pattern['stiffness_path']
    pattern_name = pattern['name']
    num_pattern_params = pattern['num_pattern_params']
    param_index = pattern['param_index']
    default_param = pattern['default_param']
    param_range = pattern['param_range']
    param_normalization_factor = pattern['param_normalization_factor']
    fusing_curve_polyline = pattern['fusing_curve_polyline_function']

    with open(experiment_file, 'r') as fp:
        data = json.load(fp)

    df = pd.DataFrame(data['data'])
    valid_tags = np.array(df['name'][df['Planar equilibrium'] == 1])


    bending_stiffness_data, stretching_stiffness_data, scale_factor_data, used_tags = visualize_stiffness.plot_all_data(kappa_path, stiffness_path, pattern_name, valid_tags, plot_data = False)

    max_bending_stiffness = np.max(bending_stiffness_data, axis = 1)
    min_bending_stiffness = np.min(bending_stiffness_data, axis = 1)
    max_stretching_stiffness = np.max(stretching_stiffness_data, axis = 1)
    min_stretching_stiffness = np.min(stretching_stiffness_data, axis = 1)
    x_scale_factors, y_scale_factors = visualize_stiffness.get_axis_scale_factors(stiffness_path, pattern_name, valid_tags)
    min_scale_factors = np.min(np.concatenate((x_scale_factors.reshape(-1, 1), y_scale_factors.reshape(-1, 1)), axis = 1), axis = 1)
    max_scale_factors = np.max(np.concatenate((x_scale_factors.reshape(-1, 1), y_scale_factors.reshape(-1, 1)), axis = 1), axis = 1)
    angle_offsets = visualize_stiffness.get_max_flattening_factor_offset(stiffness_path, pattern_name, valid_tags)
    import copy
    data_collection.append([x_scale_factors, y_scale_factors, min_bending_stiffness, max_bending_stiffness, min_stretching_stiffness, max_stretching_stiffness])
    labels.append(pattern_name)

In [52]:
import numpy.linalg as la

# Need to plot the patches over the min and max scale factors, so we can get the polygon that constrain the singular values
# The scale factors we are considering during the parametrization are from the flattening, so it's the change from the inflated state to the fabricated state, hence we need to take one over the factors we have from the average deformation gradient from homogenization.
# max_scale_factor = 1 / np.array(scale_factor_data)[:, 0]
# min_scale_factor = 1 / np.array(scale_factor_data)[:, 1]

fig, ax = plt.subplots(figsize = (10, 10))

colors = ['Greys', 'Purples', 'Blues', 'Greens', 'Oranges', 'Reds',
                      'YlOrBr', 'YlOrRd', 'OrRd', 'PuRd', 'RdPu', 'BuPu',
                      'GnBu', 'PuBu', 'YlGnBu', 'PuBuGn', 'BuGn', 'YlGn']
# labels = ['Dash Line', 'Double Dash Line', 'Cosine Curve', 'Ellipse holes angle', 'Ellipse holes width', 'Ellipse fused', 'Random Voronoi', 'Ellipse holes angle width']
for i in range(len(data_collection)):
    curr_data = data_collection[i]
    
    [x_scale_factors, y_scale_factors, min_bending_stiffness, max_bending_stiffness, min_stretching_stiffness, max_stretching_stiffness] = curr_data
    # plt.scatter(x_scale_factor, y_scale_factor, label = data_info[i][1], s = 50, alpha = 0.3)
    # plt.scatter(y_scale_factor, x_scale_factor, label = data_info[i][1], s = 50, alpha = 0.3)

    # plt.scatter(y_scale_factor, x_scale_factor, label = data_info[i][1], s = 50, alpha = 0.8, c = min_stiffness)

    print(max(y_scale_factors))
    points = np.concatenate((x_scale_factors.reshape((-1, 1)), y_scale_factors.reshape((-1, 1))), axis = 1)
    hull = ConvexHull(points)

    # for simplex in hull.simplices:
    #     plt.plot(points[simplex, 0], points[simplex, 1], 'k-')

    ax.title.set_text("Scale factors")
    plt.xlabel("x scale factors")
    plt.ylabel("y scale factors")

    plt.scatter(x_scale_factors, y_scale_factors, label = '{}_min_stiffness'.format(labels[i]), s = 200, alpha = 0.4, cmap = colors[i])
    # plt.scatter(x_scale_factors, y_scale_factors, label = '{}_min_stiffness'.format(labels[i]), s = 200, alpha = 1, c = min_bending_stiffness, cmap = colors[i])
    # plt.scatter(x_scale_factors, y_scale_factors, label = 'max_stiffness', s = 200, alpha = 1, c = max_bending_stiffness)

# Plot x = y line
lims = [
np.min([ax.get_xlim(), ax.get_ylim()]),  # min of both axes
np.max([ax.get_xlim(), ax.get_ylim()]),  # max of both axes
]

# now plot both limits against eachother
ax.plot(lims, lims, 'k-', alpha=0.75, zorder=0)
ax.set_aspect('equal')
ax.set_xlim(lims)
ax.set_ylim(lims)
ax.legend()
fig.tight_layout()
plt.savefig('scale_factor_values_{}.png'.format(name), dpi = 300)

In [56]:
import numpy.linalg as la

# Need to plot the patches over the min and max scale factors, so we can get the polygon that constrain the singular values
# The scale factors we are considering during the parametrization are from the flattening, so it's the change from the inflated state to the fabricated state, hence we need to take one over the factors we have from the average deformation gradient from homogenization.
# max_scale_factor = 1 / np.array(scale_factor_data)[:, 0]
# min_scale_factor = 1 / np.array(scale_factor_data)[:, 1]

fig, ax = plt.subplots(figsize = (10, 10))

colors = ['Greys', 'Purples', 'Blues', 'Greens', 'Oranges', 'Reds',
                      'YlOrBr', 'YlOrRd', 'OrRd', 'PuRd', 'RdPu', 'BuPu',
                      'GnBu', 'PuBu', 'YlGnBu', 'PuBuGn', 'BuGn', 'YlGn']
# labels = ['Dash Line', 'Double Dash Line', 'Cosine Curve', 'Ellipse holes angle', 'Ellipse holes width', 'Ellipse fused', 'Random Voronoi', 'Ellipse holes angle width']
for i in range(len(data_collection)):
    curr_data = data_collection[i]
    
    [x_scale_factors, y_scale_factors, min_bending_stiffness, max_bending_stiffness, min_stretching_stiffness, max_stretching_stiffness] = curr_data

    points = np.concatenate((x_scale_factors.reshape((-1, 1)), y_scale_factors.reshape((-1, 1))), axis = 1)
    hull = ConvexHull(points)

    # for simplex in hull.simplices:
    #     plt.plot(points[simplex, 0], points[simplex, 1], 'k-')

    ax.title.set_text("Scale factors")
    plt.xlabel("x scale factors")
    plt.ylabel("y scale factors")

    plt.scatter(max_bending_stiffness, min_bending_stiffness, label = '{}_min_stiffness'.format(labels[i]), s = 200, alpha = 0.4, cmap = colors[i])


ax.legend()
fig.tight_layout()
plt.savefig('scale_factor_values_{}.png'.format(name), dpi = 300)